# Exercise 2: Classifier, LoRA ranks and partial fine-tuning

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Pretrained Language Models · version 0.1.0*

> Compare classifier-only training, full fine-tuning and several LoRA ranks; report any frozen modules explicitly. Use the same real train/dev/test splits and training budget. Report macro-F1, class support, trainable parameters, peak memory and measured time.

**Setup.** `pip install torch transformers peft scikit-learn psutil requests "turkicnlp==0.5.0"`. Models are loaded at the Hugging Face revisions pinned in `MODELS` below, from a local Hugging Face cache (`HF_HUB_DIR`, default `~/.cache/huggingface/hub`), from a folder of copies (`MODELS_DIR`) or from the Hub. Data come from commit-pinned GitHub repositories: SIB-200 topic classification (seven topics; FLORES-200 sentences, so the same 1,004 sentences exist in every language), Kardeş-NLU's Kazakh XCOPA and the BERTurk repository's published evaluations. Everything runs on the CPU. Training results are cached per configuration in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes; record elapsed time and thread count for each run.

All settings fine-tune Glot500-base on the Kazakh SIB-200 training split (701 sentences, seven topics) for ten epochs with batch size 16, a linear schedule with 10% warm-up and the learning rates of `code/remaining/classify.py` (2e-4 for LoRA, 2e-5 for partial fine-tuning; 1e-3 for the classifier alone). The epoch with the best development macro-F1 is restored and evaluated once on the test split. LoRA uses rank *r* on `query` and `value` with α = 2*r*, as in the Hands-On. The saved mode named `full` freezes word embeddings and is therefore partial fine-tuning; a true full-model comparison remains to be run.

In [ ]:
import collections, csv, hashlib, io, json, math, os, random, re, statistics, threading, time
from pathlib import Path
import requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
BERTURK = "https://raw.githubusercontent.com/stefan-it/turkish-bert/0ab49e288d07f0aa40bcc1c28b655c80c1350fee/"
MODELS = {   # Hugging Face model -> pinned revision
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",
    "Qwen/Qwen2.5-1.5B-Instruct": "989aa7980e4cf806f80c7fef2b1adb7bc71aa306",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def model_source(name):
    """(path or name, revision) for from_pretrained: a local snapshot of the pinned revision if present."""
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang, split):
    """SIB-200 rows (index_id, topic, text) for a FLORES language code such as kaz_Cyrl."""
    text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
    return [(r["index_id"], r["category"], r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]

LABELS = ["entertainment", "geography", "health", "politics", "science/technology", "sports", "travel"]
GLOT = "cis-lmu/glot500-base"

_CTS = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence (TurkicNLP transliterator); Turkish is unchanged."""
    if lang not in _CTS:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        src = {"tur": Script.LATIN}.get(lang, Script.CYRILLIC)
        _CTS[lang] = Transliterator(lang, src, Script.COMMON_TURKIC)
    return _CTS[lang].transliterate(text)

def macro_f1(gold, pred):
    from sklearn.metrics import f1_score
    return f1_score(gold, pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)

class PeakRSS:
    """Peak resident memory of this process while the block runs (sampled every 0.2 s)."""
    def __enter__(self):
        import psutil
        self.p, self.peak, self.stop = psutil.Process(), 0, False
        def sample():
            while not self.stop:
                self.peak = max(self.peak, self.p.memory_info().rss); time.sleep(0.2)
        self.t = threading.Thread(target=sample, daemon=True); self.t.start(); return self
    def __exit__(self, *exc):
        self.stop = True; self.t.join()

def run(mode="lora", r=8, train="kaz_Cyrl", script="native", seed=1, epochs=10, new_tokens=0,
        tests=(("kaz_Cyrl", "native"),)):
    """Fine-tune Glot500-base for SIB-200 topics; select the epoch on the training language's dev
    macro-F1; return test macro-F1/accuracy for each (language, script) in `tests`. Cached on disk.
    mode: "head" (classifier only), "lora" (query/value, rank r, alpha 2r, classifier trained),
    "full" (all weights except the word embeddings). new_tokens > 0 adds that many Kazakh words to
    the tokenizer (see add_kazakh_words) and trains their embedding rows together with LoRA."""
    cfg = dict(base=GLOT, rev=MODELS[GLOT], mode=mode, r=r if mode == "lora" else None, train=train,
               script=script, seed=seed, epochs=epochs, new_tokens=new_tokens, tests=[list(t) for t in tests],
               batch=16, lr={"head": 1e-3, "lora": 2e-4, "full": 2e-5}[mode], warmup=0.1, v=1)
    key = hashlib.sha1(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:16]
    path = CACHE / "ch12-runs" / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text())
    import torch
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(seed); torch.set_num_threads(THREADS)
    src, rev = model_source(GLOT)
    tok = tokenizer(GLOT)
    model = AutoModelForSequenceClassification.from_pretrained(
        src, revision=rev, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)),
        label2id={l: i for i, l in enumerate(LABELS)})
    added = add_kazakh_words(tok, model, new_tokens) if new_tokens else []
    if mode == "full":
        model.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
    elif mode == "head":
        for n, p in model.named_parameters():
            p.requires_grad_(n.startswith("classifier"))
    else:
        from peft import LoraConfig, TaskType, get_peft_model
        extra = {"trainable_token_indices": {"word_embeddings": added}} if added else {}
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
                                                 target_modules=["query", "value"], modules_to_save=["classifier"], **extra))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    def data(lang, split, scr):
        rows = sib(lang, split)
        texts = [t if scr == "native" else to_cts(lang.split("_")[0], t) for _, _, t in rows]
        return texts, [LABELS.index(c) for _, c, _ in rows]
    def predict(texts):
        model.eval(); out = []
        with torch.inference_mode():
            for i in range(0, len(texts), 32):
                enc = tok(texts[i:i + 32], padding=True, truncation=True, max_length=256, return_tensors="pt")
                out += model(**enc).logits.argmax(-1).tolist()
        return out
    tr_x, tr_y = data(train, "train", script)
    dev_x, dev_y = data(train, "dev", script)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.01)
    steps = epochs * math.ceil(len(tr_x) / cfg["batch"])
    sched = get_linear_schedule_with_warmup(opt, int(cfg["warmup"] * steps), steps)
    rng = random.Random(seed); best = (-1, None, None); history = []
    t0 = time.time()
    with PeakRSS() as mem:
        for ep in range(1, epochs + 1):
            model.train(); order = list(range(len(tr_x))); rng.shuffle(order)
            for i in range(0, len(order), cfg["batch"]):
                b = order[i:i + cfg["batch"]]
                enc = tok([tr_x[j] for j in b], padding=True, truncation=True, max_length=256, return_tensors="pt")
                loss = model(**enc, labels=torch.tensor([tr_y[j] for j in b])).loss
                loss.backward(); opt.step(); sched.step(); opt.zero_grad()
            f1 = macro_f1(dev_y, predict(dev_x)); history.append(round(f1, 4))
            if f1 > best[0]:
                best = (f1, ep, {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad})
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best[2]:
                    p.copy_(best[2][n])
        results = {}
        for lang, scr in tests:
            x, y = data(lang, "test", scr); pred = predict(x)
            results[f"{lang}/{scr}"] = {"macro_f1": macro_f1(y, pred), "accuracy": sum(a == b for a, b in zip(y, pred)) / len(y),
                                        "n": len(y), "pred": pred}
    res = dict(cfg=cfg, trainable=trainable, total=total, dev_history=history, best_epoch=best[1],
               dev_macro_f1=best[0], tests=results, seconds=time.time() - t0, peak_rss_gb=mem.peak / 1e9,
               added_tokens=added)
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(res))
    return res

PROBE = r"""
import json, sys, threading, time, psutil, torch
a = json.loads(sys.argv[1]); torch.set_num_threads(a["threads"]); torch.manual_seed(0)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
proc = psutil.Process(); start = proc.memory_info().rss
tok = AutoTokenizer.from_pretrained(a["path"]); m = AutoModelForSequenceClassification.from_pretrained(a["path"], num_labels=7)
if a["mode"] == "full":
    m.roberta.embeddings.word_embeddings.weight.requires_grad_(False)
elif a["mode"] == "head":
    for n, p in m.named_parameters(): p.requires_grad_(n.startswith("classifier"))
else:
    from peft import LoraConfig, TaskType, get_peft_model
    m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=a["r"], lora_alpha=2 * a["r"], lora_dropout=0.1,
                                     target_modules=["query", "value"], modules_to_save=["classifier"]))
loaded = proc.memory_info().rss; peak = [loaded]; stop = []
def sample():
    while not stop: peak[0] = max(peak[0], proc.memory_info().rss); time.sleep(0.05)
t = threading.Thread(target=sample, daemon=True); t.start()
opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4)
enc = tok(a["texts"], padding=True, return_tensors="pt"); y = torch.zeros(len(a["texts"]), dtype=torch.long)
t0 = time.time()
for _ in range(a["steps"]):
    m(**enc, labels=y).loss.backward(); opt.step(); opt.zero_grad()
sec = (time.time() - t0) / a["steps"]; stop.append(1); t.join()
print(json.dumps({"start": start, "loaded": loaded, "peak": peak[0], "sec_per_step": sec}))
"""

def peak_memory(mode, r=8, steps=5):
    """Peak resident memory of a fresh process that loads Glot500 and takes `steps` training steps
    on the first 16 Kazakh training sentences (batch 16), and its seconds per step. Cached."""
    import subprocess, sys
    key = f"{mode}-{r if mode == 'lora' else ''}-{steps}-{THREADS}"
    path = CACHE / "ch12-memory.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    if key not in saved:
        texts = [t for _, _, t in sib("kaz_Cyrl", "train")[:16]]
        args = dict(path=model_source(GLOT)[0], mode=mode, r=r, steps=steps, threads=THREADS, texts=texts)
        out = subprocess.run([sys.executable, "-c", PROBE, json.dumps(args, ensure_ascii=False)],
                             capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
        saved[key] = json.loads(out); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(saved))
    return saved[key]

def kazakh_candidates(tok, n):
    """The n most frequent Kazakh word forms of the SIB-200 training split that Glot500 splits into
    three or more pieces (ties broken alphabetically)."""
    words = collections.Counter(w for _, _, t in sib("kaz_Cyrl", "train") for w in re.findall(r"\w+", t))
    long = [w for w, c in sorted(words.items(), key=lambda x: (-x[1], x[0]))
            if len(tok.tokenize(" " + w)) >= 3]
    return long[:n]

def add_kazakh_words(tok, model, n):
    """Add whole-word tokens without changing old IDs; initialise each new row as the mean of the
    old embeddings of the word's pieces. Returns the new IDs."""
    import torch
    from tokenizers import AddedToken
    words = kazakh_candidates(tok, n)
    old_len = len(tok)
    pieces = {w: tok(" " + w, add_special_tokens=False)["input_ids"] for w in words}
    tok.add_tokens([AddedToken(w, single_word=True, lstrip=True) for w in words])
    model.resize_token_embeddings(len(tok), mean_resizing=False)
    emb = model.get_input_embeddings().weight
    with torch.no_grad():
        for w in words:
            emb[tok.convert_tokens_to_ids(w)] = emb[pieces[w]].mean(0)
    return list(range(old_len, len(tok)))

def wilson(k, n, z=1.959963984540054):
    p = k / n; den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

print("models:", ", ".join(f"{n} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in MODELS))

In [ ]:
CONFIGS = [("head", None, 1), ("lora", 2, 1), ("lora", 8, 1), ("lora", 8, 2), ("lora", 8, 3), ("lora", 32, 1), ("full", None, 1)]
results = {}
for mode, r, seed in CONFIGS:
    results[mode, r, seed] = run(mode=mode, r=r or 8, seed=seed)
test_rows = sib("kaz_Cyrl", "test")
print("SIB-200 Kazakh: train", len(sib("kaz_Cyrl", "train")), "dev", len(sib("kaz_Cyrl", "dev")), "test", len(test_rows))
print("test support:", dict(sorted(collections.Counter(c for _, c, _ in test_rows).items())))
print(f"{'setting':<14}{'seed':>5}{'trainable':>13}{'share':>8}{'epoch':>6}{'dev F1':>8}{'test F1':>8}{'test acc':>9}{'minutes':>8}")
for (mode, r, seed), res in results.items():
    t = res["tests"]["kaz_Cyrl/native"]
    name = mode if mode != "lora" else f"LoRA r={r}"
    print(f"{name:<14}{seed:>5}{res['trainable']:>13,}{res['trainable'] / res['total']:>8.2%}{res['best_epoch']:>6}"
          f"{res['dev_macro_f1']:>8.3f}{t['macro_f1']:>8.3f}{t['accuracy']:>9.3f}{res['seconds'] / 60:>8.1f}")
print("memory, each setting in a fresh process (5 training steps, batch 16):")
for mode, r in (("head", None), ("lora", 2), ("lora", 8), ("lora", 32), ("full", None)):
    m = peak_memory(mode, r or 8)
    print(f"  {mode if mode != 'lora' else f'LoRA r={r}':<10} after loading {m['loaded'] / 1e9:.2f} GB, peak while training "
          f"{m['peak'] / 1e9:.2f} GB, {m['sec_per_step']:.1f} s per step")
f1s = [results["lora", 8, s]["tests"]["kaz_Cyrl/native"]["macro_f1"] for s in (1, 2, 3)]
print(f"LoRA r=8 over three seeds: test macro-F1 {statistics.mean(f1s):.3f} (range {min(f1s):.3f}-{max(f1s):.3f})")
r8 = results["lora", 8, 1]
print(f"LoRA r=8 trainable = 12 layers x 2 projections x 8 x (768 + 768) = {12 * 2 * 8 * 1536:,} adapter weights "
      f"+ classifier {r8['trainable'] - 12 * 2 * 8 * 1536:,}")
gold = [LABELS.index(c) for _, c, _ in test_rows]
pred = r8["tests"]["kaz_Cyrl/native"]["pred"]
from sklearn.metrics import f1_score
print("per-class test F1 (LoRA r=8, seed 1):", {LABELS[i]: round(float(v), 2) for i, v in
      enumerate(f1_score(gold, pred, labels=list(range(len(LABELS))), average=None, zero_division=0))})

## Worked solution

Output of `experiments/ch12_exercises/run_ch12_exercises.py`:

```
SIB-200 Kazakh: train 701 dev 99 test 204
test support: {'entertainment': 19, 'geography': 17, 'health': 22, 'politics': 30, 'science/technology': 51, 'sports': 25, 'travel': 40}
setting        seed    trainable   share epoch  dev F1 test F1 test acc minutes
head              1      595,975   0.15%     8   0.311   0.365    0.510     8.5
LoRA r=2          1      669,703   0.17%     7   0.806   0.829    0.838    15.2
LoRA r=8          1      890,887   0.23%    10   0.807   0.807    0.819    15.2
LoRA r=8          2      890,887   0.23%    10   0.817   0.861    0.868    15.1
LoRA r=8          3      890,887   0.23%     9   0.804   0.843    0.848    15.1
LoRA r=32         1    1,775,623   0.45%    10   0.835   0.838    0.848    15.5
full              1   86,047,495  21.83%     7   0.828   0.792    0.804    20.2
memory, each setting in a fresh process (5 training steps, batch 16):
  head       after loading 1.27 GB, peak while training 1.70 GB, 1.3 s per step
  LoRA r=2   after loading 1.29 GB, peak while training 2.29 GB, 1.9 s per step
  LoRA r=8   after loading 1.29 GB, peak while training 2.30 GB, 1.5 s per step
  LoRA r=32  after loading 1.29 GB, peak while training 2.32 GB, 1.4 s per step
  full       after loading 1.28 GB, peak while training 2.96 GB, 3.0 s per step
LoRA r=8 over three seeds: test macro-F1 0.837 (range 0.807-0.861)
LoRA r=8 trainable = 12 layers x 2 projections x 8 x (768 + 768) = 294,912 adapter weights + classifier 595,975
per-class test F1 (LoRA r=8, seed 1): {'entertainment': 0.67, 'geography': 0.79, 'health': 0.8, 'politics': 0.88, 'science/technology': 0.86, 'sports': 0.88, 'travel': 0.78}
```

**The training settings.** Classifier-only training gives test macro-F1 0.365 on 204 Kazakh sentences. LoRA runs give 0.807–0.861. The row labelled `full` is actually partial fine-tuning: it freezes the 308 million word-embedding parameters and updates the remaining encoder weights and classifier, 86 million parameters. That run scores 0.792. These are outcomes of the chosen recipes, not proof that frozen representations cannot separate topics or that LoRA generally outperforms full fine-tuning.

**Seeds and ranks.** Rank 8 gives 0.807, 0.861 and 0.843 across three seeds. Ranks 2 and 32 each have one run. The observed seed range exceeds the rank difference in those single runs; use repeated runs for every setting before ranking them.

**Parameters and cost.** Rank 8 trains 294,912 adapter weights and 595,975 classifier parameters. The separate five-step CPU probe records roughly 2.3 GB peak resident memory for LoRA and 3.0 GB for partial fine-tuning. These are process-memory measurements, not GPU requirements. The table’s 15 and 20 minutes describe complete training runs including development evaluation, not individual epochs.

**Protocol.** Checkpoints are selected on 99 development sentences. The runner tokenises with truncation at 256 tokens; audit affected examples before treating these scores as full-input results. Per-class support ranges from 17 to 51 test items, so retain per-class scores alongside macro-F1. The separate hands-on runner rejects overlength input rather than truncating it.

## Your turn — extension

1. Run ranks 2 and 32 with two more seeds and decide whether any rank difference survives.
2. Try LoRA on all linear layers (`target_modules=["query", "key", "value", "dense"]`) with the same budget.